🤖 Machine Learning &nbsp;·&nbsp; *Bài 18/23*

[« Machine Learning - Grid Search](017_ml_grid_search.ipynb) &nbsp;•&nbsp; [📚 Mục lục](../README.md) &nbsp;•&nbsp; [Machine Learning - K-means »](019_ml_k_means.ipynb)

# Machine Learning - Preprocessing - Categorical Data

> 📖 **Học song song trên W3Schools:** [python_ml_preprocessing.asp](https://www.w3schools.com/python/python_ml_preprocessing.asp)
>
> 🎯 **Trong bài này:** Categorical Data · One Hot Encoding · Predict CO2 · Dummifying

## Categorical Data — Dữ liệu phân loại

Khi dữ liệu có các nhóm được biểu diễn bằng **chuỗi**, rất khó dùng chúng để huấn luyện mô hình — vốn thường chỉ nhận dữ liệu **số**.

Thay vì bỏ qua dữ liệu phân loại và mất thông tin, ta có thể **biến đổi** dữ liệu để dùng được trong mô hình.

Xem lại bảng xe dùng ở bài [Multiple Regression](010_ml_multiple_regression.ipynb):

In [1]:
import pandas as pd

cars = pd.read_csv('cars.csv')
print(cars.to_string())

           Car       Model  Volume  Weight  CO2
0         Ford      Mondeo    1600    1559  103
1         Ford       B-Max    1600    1232   99
2     Mercedes         SLK    2500    1386  111
3         Ford      Fiesta    1000    1079   99
4     Mercedes     E-Class    2100    1619  106
5     Mercedes         CLA    1500    1524  101
6        Honda       Civic    1600    1186  100
7      Hyundai         I20    1100    1037   93
8        Skoda       Fabia    1000    1124   94
9         Audi          A4    2000    1509  105
10        Mini      Cooper    1500    1134  101
11        Ford      Fiesta    1500    1164   98
12        Audi          A1    1600    1092   93
13         BMW           2    1600    1430  101
14       Mazda           3    2200    1261  109
15       Skoda      Citigo    1000     949   89
16       Volvo         S60    2000    1385  100
17        Audi          A6    2000    1754  106
18         BMW           1    1600    1406  108
19        Opel      Zafira    1600    13

Ở bài Multiple Regression, ta dự đoán CO2 từ dung tích và trọng lượng, nhưng **bỏ qua** thông tin hãng xe và mẫu xe.

Hãng xe (`Car`) hay mẫu xe (`Model`) có thể giúp dự đoán CO2 tốt hơn.

## One Hot Encoding

Không thể dùng trực tiếp cột `Car` hay `Model` vì chúng không phải số. Không có quan hệ tuyến tính giữa biến phân loại `Car` (chuỗi) và CO2 (số).

Để giải quyết, cần biểu diễn biến phân loại bằng số. Một cách là có **một cột cho mỗi nhóm** trong biến phân loại. Với mỗi cột, giá trị là **1** nếu hàng thuộc nhóm đó, ngược lại là **0**.

Cách làm này gọi là **One Hot Encoding**. Không cần làm thủ công — Pandas có hàm `get_dummies()`:

In [2]:
import pandas as pd

cars = pd.read_csv('cars.csv')
ohe_cars = pd.get_dummies(cars[['Car']], dtype=int)

print(ohe_cars.to_string())

    Car_Audi  Car_BMW  Car_Fiat  Car_Ford  Car_Honda  Car_Hyundai  Car_Mazda  Car_Mercedes  Car_Mini  Car_Mitsubishi  Car_Opel  Car_Skoda  Car_Suzuki  Car_Toyota  Car_VW  Car_Volvo
0          0        0         0         1          0            0          0             0         0               0         0          0           0           0       0          0
1          0        0         0         1          0            0          0             0         0               0         0          0           0           0       0          0
2          0        0         0         0          0            0          0             1         0               0         0          0           0           0       0          0
3          0        0         0         1          0            0          0             0         0               0         0          0           0           0       0          0
4          0        0         0         0          0            0          0             1     

Kết quả: một cột được tạo cho **mỗi hãng xe** trong cột `Car`.

## Predict CO2 — Dự đoán CO2

Dùng thông tin bổ sung này cùng dung tích và trọng lượng để dự đoán CO2.

Để kết hợp thông tin, dùng hàm `concat()` của Pandas:

In [3]:
from sklearn import linear_model

X = pd.concat([cars[['Volume', 'Weight']], ohe_cars], axis=1)
y = cars['CO2']

regr = linear_model.LinearRegression()
regr.fit(X, y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


Giờ có thể dự đoán CO2 dựa trên dung tích, trọng lượng **và hãng xe**.

Dự đoán CO2 của một chiếc **VW** nặng 2300 kg, dung tích 1300 cm³:

In [4]:
new_car = pd.DataFrame(0, index=[0], columns=X.columns)
new_car.loc[0, ["Volume", "Weight", "Car_VW"]] = [1300, 2300, 1]

predictedCO2 = regr.predict(new_car)
print(predictedCO2)

[112.17251364]


## Dummifying

Không cần tạo **một cột cho mỗi nhóm**. Thông tin có thể giữ nguyên với số cột **ít hơn số nhóm 1 cột**.

Ví dụ biến màu `red` / `blue`: chỉ cần **một** cột — giá trị 1 nghĩa là `blue`, 0 nghĩa là `red`:

In [5]:
import pandas as pd

colors = pd.DataFrame({'color': ['blue', 'red']})
print(colors)

dummies = pd.get_dummies(colors, drop_first=True, dtype=int)
print(dummies)

  color
0  blue
1   red
   color_red
0          0
1          1


Nếu có **hơn 2** nhóm:

In [6]:
import pandas as pd

colors = pd.DataFrame({'color': ['blue', 'red', 'green']})
dummies = pd.get_dummies(colors, drop_first=True, dtype=int)
dummies['color'] = colors['color']

print(dummies)

   color_green  color_red  color
0            0          0   blue
1            0          1    red
2            1          0  green


Cột bị bỏ (`blue`) chính là khi **mọi** cột còn lại đều bằng 0.

---

## 🧠 Ghi nhớ nhanh

> - Mô hình cần **số** → biến chữ (categorical) phải mã hoá.
> - **One Hot Encoding**: mỗi nhóm một cột 0/1 — `pd.get_dummies(df[['col']], dtype=int)`.
> - Ghép với các cột số: `pd.concat([...], axis=1)`.
> - **Dummifying**: `drop_first=True` → bớt 1 cột mà không mất thông tin.

---

## 📝 Exercise — Bài tập nhanh

**❓ Câu hỏi:** Hàm Pandas nào thực hiện One Hot Encoding?

- **A.** `pd.get_dummies()`
- **B.** `pd.one_hot()`
- **C.** `pd.categorical()`
- **D.** `pd.encode()`

<details>
<summary>👉 Xem đáp án</summary>

✅ **Đáp án: A** — `pd.get_dummies()`

</details>

---

[« Machine Learning - Grid Search](017_ml_grid_search.ipynb) &nbsp;•&nbsp; [📚 Mục lục](../README.md) &nbsp;•&nbsp; [Machine Learning - K-means »](019_ml_k_means.ipynb)